# CS781 Phase 7 — Experiment 5: Model 1 + Full Fine-Tuning

**Retrieval strategy:** BM25 candidate generation + Model 1 cross-encoder reranking.

This experiment starts from the original Phase-1 **Model 1 (~120M parameter encoder/prefix-LM)** checkpoint and its matching LegalBPE tokenizer. It does not repeat Phase-1 pretraining and does not use any external pretrained model or legal corpus.

## Method
1. Build BM25 indexes using each split's own candidate pool.
2. Mine hard negatives from the Train candidate pool for labelled Train queries.
3. Jointly encode each query–candidate pair using the exact Model 1 architecture.
4. Fully fine-tune **all 14 transformer blocks, token embeddings, final normalization, and the ranking head**.
5. Train with a query-group listwise cross-entropy objective: one positive against five BM25 hard negatives.
6. Use gradient checkpointing and two-query gradient accumulation to reduce activation memory on a Kaggle T4.
7. Evaluate each epoch on labelled Dev; select the best checkpoint and lexical/neural fusion weight using Dev Micro-F1@7.
8. Compare the selected reranker against BM25-only on Dev. Use the better Dev method for Test.
9. Validate the exact submission schema and ensure `prediction.json` is at the ZIP root.

## Full fine-tuning configuration
- 5 epochs (early stopping, patience 2); 448-token pair context (128 query + 300 candidate tokens).
- BM25 top-100 candidate retrieval; 5 hard negatives per training query group.
- Top transformer block LR `2e-5`, layer-wise decay `0.85`, embedding LR `3e-6`, scoring-head LR `1.6e-4`.
- Up to 2 distinct positives per query per epoch, weight EMA (0.995), backbone dropout 0.1, extended fusion grid.
- FP16 AMP, gradient clipping, warmup + cosine decay.
- Dev fusion grid searched after every epoch; no Test labels are accessed.

**Improvements over the original run (core implementation unchanged):** fixed the `checkpoint` name-clash bug that would crash gradient checkpointing; more positive supervision per epoch; EMA weights; light dropout; higher LRs for full FT; fusion grid extended beyond the old edge value (2.0); early stopping.

**Important:** Full fine-tuning is a hypothesis, not a guaranteed improvement. The official Test score is unknown until Codabench evaluates the submission. The prior best official score remains **0.3473**.

**Outputs:** `/kaggle/working/phase7_model1_full_ft/` — all files directly in this one folder, no subfolders.


In [1]:
# Cell 1 — Runtime, paths, Model 1 checkpoint/tokenizer discovery, and Phase-7 data
import os, sys, glob, json, time, math, random, re, gc, zipfile, warnings, heapq
from pathlib import Path
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint  # FIX: was never imported, and later shadowed by the loaded ckpt dict
from contextlib import contextmanager
from huggingface_hub import hf_hub_download

SEED = 1702
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

def stamp():
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d %H:%M:%S IST")

print("="*78)
print("PHASE 7 | EXPERIMENT 5 | MODEL 1 + FULL FINE-TUNING")
print("Started:", stamp(), "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
assert DEVICE.type == "cuda", "Please select a Kaggle GPU accelerator (T4 recommended)."

OUTPUT_DIR = Path("/kaggle/working/phase7_model1_full_ft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Flat output layout: every checkpoint and artifact is written directly here.
CKPT_DIR = OUTPUT_DIR

# Attach the Phase-1 Model 1 weights/tokenizer Kaggle dataset.
# Prefer the exact encoder full-run best checkpoint; never silently choose Model 3/4.
all_pt = glob.glob("/kaggle/input/**/*.pt", recursive=True) + glob.glob("/kaggle/input/**/*.pth", recursive=True)
m1_candidates = [p for p in all_pt if "enc_full_best" in os.path.basename(p).lower()]
if not m1_candidates:
    m1_candidates = [p for p in all_pt if "enc_full" in os.path.basename(p).lower() and "best" in os.path.basename(p).lower()
                     and "encdec" not in p.lower() and "model3" not in p.lower()]
if not m1_candidates:
    raise FileNotFoundError("Model 1 checkpoint not found. Attach the Phase-1 Model 1 dataset containing enc_full_best*.pt")
# Prefer a checkpoint explicitly under phase1model-wts; then shortest path.
m1_candidates = sorted(set(m1_candidates), key=lambda p: (("phase1model-wts" not in p.lower()), ("enc_full_best" not in os.path.basename(p).lower()), len(p)))
CKPT_PATH = m1_candidates[0]

tok_candidates = glob.glob("/kaggle/input/**/tokenizer (1).json", recursive=True)
if not tok_candidates:
    raise FileNotFoundError("Model 1 tokenizer.json not found. Attach the matching Phase-1 Model 1 tokenizer dataset.")
same_dir = [p for p in tok_candidates if os.path.dirname(p) == os.path.dirname(CKPT_PATH)]
m1_tok = [p for p in tok_candidates if "model1" in p.lower() or "phase1model-wts" in p.lower()]
if same_dir:
    TOKENIZER_PATH = sorted(same_dir)[0]
elif m1_tok:
    TOKENIZER_PATH = sorted(m1_tok)[0]
elif len(tok_candidates) == 1:
    TOKENIZER_PATH = tok_candidates[0]
else:
    print("Tokenizer candidates:")
    for p in tok_candidates: print(" -", p)
    raise RuntimeError("Multiple tokenizer.json files found; set TOKENIZER_PATH explicitly to Model 1 tokenizer.")

print("Model 1 checkpoint:", CKPT_PATH)
print("Tokenizer:", TOKENIZER_PATH)
assert "encdec" not in CKPT_PATH.lower() and "model3" not in CKPT_PATH.lower()

# Phase 7 dataset: same official repository and split-specific pools as previous experiment.
REPO_ID = "Exploration-Lab/CS-781-Capstone"
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = None
    print("No HF_TOKEN secret found; attempting public access.")

def load_hf_json(relative_path):
    local = hf_hub_download(repo_id=REPO_ID, filename=relative_path,
                            repo_type="dataset", token=HF_TOKEN)
    with open(local, "r", encoding="utf-8") as f:
        return json.load(f)

print("\nLoading Phase 7 splits:", stamp())
train_queries = load_hf_json("Phase_7_PCR/Train/train.json")
train_candidates = load_hf_json("Phase_7_PCR/Train/train_candidates.json")
dev_queries = load_hf_json("Phase_7_PCR/Dev/dev.json")
dev_candidates = load_hf_json("Phase_7_PCR/Dev/dev_candidates.json")
test_queries = load_hf_json("Phase_7_PCR/Test/test.json")
test_candidates = load_hf_json("Phase_7_PCR/Test/test_candidates.json")

# Normalize IDs to strings and preserve record fields.
def normalize_dict(d):
    return {str(k): v for k, v in d.items()}
train_queries=normalize_dict(train_queries); train_candidates=normalize_dict(train_candidates)
dev_queries=normalize_dict(dev_queries); dev_candidates=normalize_dict(dev_candidates)
test_queries=normalize_dict(test_queries); test_candidates=normalize_dict(test_candidates)

print(f"Train: {len(train_queries):,} queries | {len(train_candidates):,} candidates")
print(f"Dev:   {len(dev_queries):,} queries | {len(dev_candidates):,} candidates")
print(f"Test:  {len(test_queries):,} queries | {len(test_candidates):,} candidates")
assert train_queries and train_candidates and dev_queries and dev_candidates and test_queries and test_candidates

def record_text(record):
    value = record.get("text", [])
    if isinstance(value, list):
        return " ".join(str(x) for x in value)
    return str(value)

# Basic EDA before building retrieval indexes.
for split_name, queries, candidates in [
    ("Train", train_queries, train_candidates),
    ("Dev", dev_queries, dev_candidates),
    ("Test", test_queries, test_candidates),
]:
    q_lens = [len(record_text(x)) for x in queries.values()]
    c_lens = [len(record_text(x)) for x in candidates.values()]
    print(f"{split_name} EDA | q chars median/p90={int(np.median(q_lens)):,}/{int(np.percentile(q_lens,90)):,} "
          f"| candidate chars median/p90={int(np.median(c_lens)):,}/{int(np.percentile(c_lens,90)):,}")
if train_queries:
    label_counts = [len(q.get("relevant_candidates", [])) for q in train_queries.values()]
    print("Train relevant candidates/query | mean:", round(float(np.mean(label_counts)),2),
          "| median:", int(np.median(label_counts)), "| total:", sum(label_counts))


PHASE 7 | EXPERIMENT 5 | MODEL 1 + FULL FINE-TUNING
Started: 2026-11-07 08:02:15 IST | Device: cuda
GPU: Tesla T4
VRAM GB: 14.56
Model 1 checkpoint: /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt
Tokenizer: /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
HF_TOKEN loaded from Kaggle Secrets.

Loading Phase 7 splits: 2026-11-07 08:02:15 IST


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (2 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Train: 827 queries | 4,320 candidates
Dev:   118 queries | 1,023 candidates
Test:  237 queries | 1,727 candidates
Train EDA | q chars median/p90=25,893/72,937 | candidate chars median/p90=20,848/71,744
Dev EDA | q chars median/p90=26,363/88,437 | candidate chars median/p90=23,207/97,249
Test EDA | q chars median/p90=26,411/82,525 | candidate chars median/p90=22,627/78,000
Train relevant candidates/query | mean: 6.42 | median: 4 | total: 5308


In [2]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

# Load the matching Model 1 tokenizer; vocabulary must match Model 1's embedding matrix.
tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
print("Loaded Model 1 tokenizer | vocab:", len(tokenizer))
print("Tokenizer round-trip:", tokenizer.decode(tokenizer.encode("Supreme Court of India — Section 14 ₹100")) ==
      "Supreme Court of India — Section 14 ₹100")
assert len(tokenizer) == 24576, f"Unexpected Model 1 vocab size: {len(tokenizer)}"


Tokenizer class ready.
Loaded Model 1 tokenizer | vocab: 24576
Tokenizer round-trip: True


In [3]:
# Cell 3 — Retrieval/training configuration (Model 1 is larger than Model 3)
VOCAB_SIZE = 24576
D_MODEL = 768
N_LAYERS = 14
N_HEADS = 12
FFN_DIM = 3072
DROPOUT = 0.1               # light regularisation (runtime-only; no effect on checkpoint weights)
MAX_POSITIONS = 2048

# Joint query-candidate pair layout.
PAIR_MAX_LEN = 448          # longer pair context (was 352)
QUERY_TOKENS = 128
CANDIDATE_TOKENS = 300      # was 220
HARD_NEGATIVE_POOL = 100
TRAIN_NEGATIVES = 5         # keep hard negatives, limit pair batch on T4
RERANK_TOP_N = 100
TOP_K = 7

# Full fine-tuning: all 14 blocks + token embeddings + final norm + scorer.
EPOCHS = 5
POS_PER_QUERY = 2           # distinct positives sampled per query per epoch (each still 1 pos vs 5 hard negs)
EMA_DECAY = 0.995           # weight-EMA used for Dev eval / checkpoints / Test
PATIENCE = 2                # stop if Dev F1 doesn't improve for this many epochs
QUERY_GROUP_BATCH = 1       # one query group (6 pairs) per forward pass for T4 memory safety
ACCUM_STEPS = 2             # accumulate two query groups before each optimizer update
LEARNING_RATE = 2.0e-5      # top transformer block LR; lower layers use layer-wise decay
EMBEDDING_LR = 3.0e-6       # conservative LR for the pretrained token embedding matrix
LAYER_LR_DECAY = 0.85       # lower transformer blocks receive progressively smaller LRs
HEAD_LR_MULT = 8.0          # scoring head learns faster than the backbone
WEIGHT_DECAY = 0.02
WARMUP_FRAC = 0.06
EVAL_BATCH_SIZE = 8         # lower eval batch for the 120M full-finetuned model on T4
FUSION_GRID = [0.0, 0.03, 0.06, 0.10, 0.15, 0.20, 0.30, 0.45, 0.60, 0.80, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 10.0]  # extended: old best sat at grid edge (2.0)
GRAD_CLIP = 0.8
USE_AMP = True
GRAD_CHECKPOINTING = True

print("Model 1 config:", {
    "parameters_expected": "~120M", "layers": N_LAYERS, "hidden": D_MODEL,
    "pair_max_len": PAIR_MAX_LEN, "query_tokens": QUERY_TOKENS,
    "candidate_tokens": CANDIDATE_TOKENS, "train_negatives": TRAIN_NEGATIVES,
    "epochs": EPOCHS, "pos_per_query": POS_PER_QUERY, "ema": EMA_DECAY, "dropout": DROPOUT, "fine_tuning": "FULL", "accum_steps": ACCUM_STEPS,
    "started_at": stamp()
})


Model 1 config: {'parameters_expected': '~120M', 'layers': 14, 'hidden': 768, 'pair_max_len': 448, 'query_tokens': 128, 'candidate_tokens': 300, 'train_negatives': 5, 'epochs': 5, 'pos_per_query': 2, 'ema': 0.995, 'dropout': 0.1, 'fine_tuning': 'FULL', 'accum_steps': 2, 'started_at': '2026-11-07 08:02:32 IST'}


In [4]:
# Cell 9 — Optimized ~120M Prefix Encoder LM with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class PrefixSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B,T,C = x.shape
        qkv = self.qkv(x).view(B,T,3,self.n_heads,self.head_dim)
        q,k,v = qkv.unbind(dim=2)
        q = q.transpose(1,2)
        k = k.transpose(1,2)
        v = v.transpose(1,2)

        if past_kv is not None:
            pk,pv = past_kv
            k = torch.cat([pk,k], dim=2)
            v = torch.cat([pv,v], dim=2)

        y = F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.out_proj(y)

        new_kv = (k,v) if use_cache else None
        return y, new_kv


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixSelfAttention(d_model,n_heads,dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model,ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim,d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        a,new_kv = self.attn(
            self.ln1(x),
            attn_mask=attn_mask,
            past_kv=past_kv,
            use_cache=use_cache,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x,new_kv


class PrefixEncoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size,d_model)
        self.pos = SinusoidalPositionalEncoding(d_model,max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            EncoderBlock(d_model,n_heads,ffn_dim,dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        # Tied output projection: no second vocab-sized matrix.
        self.apply(self._init_weights)

        # GPT-style residual projection scaling improves deep-model stability.
        scaled_std = 0.02 / math.sqrt(2*n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    @staticmethod
    def prefix_mask(T, prefix_len, device):
        # True means attention is allowed for SDPA boolean mask.
        q = torch.arange(T, device=device)[:,None]
        k = torch.arange(T, device=device)[None,:]
        p = int(prefix_len)
        allowed = ((q < p) & (k < p)) | ((q >= p) & (k <= q))
        return allowed[None,None,:,:]

    def forward_hidden(self, input_ids, prefix_len=None, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        # position_ids / key_valid are only used for LEFT-PADDED BATCHED generation; when both are None the
        # behaviour is exactly the original (training path unchanged).
        B,T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len+T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len+T, device=input_ids.device
            )[None,:].expand(B,T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        # Full training sequence uses the prefix-LM mask.
        # Initial generation prefill has prefix_len == T, therefore it is fully bidirectional.
        # Cached one-token decoding needs no mask because only past/current KVs exist.
        if key_valid is not None:
            # (B,1,1,S) bool, True = real token. Prompt is bidirectional; padding is never attended to.
            attn_mask = key_valid[:,None,None,:]
        elif past_key_values is None and prefix_len is not None and int(prefix_len) < T:
            attn_mask = self.prefix_mask(T,prefix_len,input_ids.device)
        else:
            attn_mask = None

        new_cache = [] if use_cache else None

        for i,block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            if (GRAD_CHECKPOINTING and self.training and torch.is_grad_enabled()
                    and not use_cache and past_key_values is None):
                # Save activation memory while retaining gradients through every block.
                x = grad_checkpoint(
                    lambda z, _block=block, _mask=attn_mask: _block(
                        z, attn_mask=_mask, past_kv=None, use_cache=False
                    )[0],
                    x,
                    use_reentrant=False
                )
                kv = None
            else:
                x,kv = block(
                    x,
                    attn_mask=attn_mask,
                    past_kv=past,
                    use_cache=use_cache,
                )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x,new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)

print("Model 1 architecture classes ready.")


# Instantiate the exact Phase-1 Model 1 architecture and load its best pretrained checkpoint.
model = PrefixEncoderLM(
    vocab_size=len(tokenizer), d_model=D_MODEL, n_layers=N_LAYERS,
    n_heads=N_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT,
    max_positions=MAX_POSITIONS
).to(DEVICE)

ckpt_obj = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
base_state = ckpt_obj.get("model", ckpt_obj.get("state_dict", ckpt_obj))
missing, unexpected = model.load_state_dict(base_state, strict=False)
# No adapter modules are added to this architecture, so all base parameters must match.
assert not missing, f"Missing Model 1 checkpoint keys: {missing[:10]}"
assert not unexpected, f"Unexpected checkpoint keys: {unexpected[:10]}"
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Model 1 | params={n_params/1e6:.2f}M | checkpoint keys={len(base_state)}")
assert 100_000_000 <= n_params <= 150_000_000, f"Unexpected Model 1 parameter count: {n_params}"

# FULL fine-tuning: every pretrained Model 1 parameter is trainable.
for p in model.parameters():
    p.requires_grad = True
assert all(p.requires_grad for p in model.parameters()), "Full fine-tuning setup failed."

class PairScorer(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.12)
        self.score = nn.Sequential(
            nn.Linear(D_MODEL, 384),
            nn.GELU(),
            nn.Dropout(0.12),
            nn.Linear(384, 1)
        )
    def forward(self, input_ids, valid):
        # Every non-padding pair token belongs to the bidirectional prefix.
        h, _ = self.backbone.forward_hidden(
            input_ids, prefix_len=input_ids.shape[1], key_valid=valid
        )
        mask = valid.unsqueeze(-1).to(h.dtype)
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.score(self.dropout(pooled.float())).squeeze(-1)

pair_model = PairScorer(model).to(DEVICE)
trainable = sum(p.numel() for p in pair_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in pair_model.parameters())
del ckpt_obj; gc.collect()
print(f"Trainable parameters: {trainable/1e6:.2f}M / {total/1e6:.2f}M ({100*trainable/total:.1f}%)")
assert trainable >= 100_000_000, "Expected essentially all Model 1 parameters to be trainable."


Model 1 architecture classes ready.
Loaded Model 1 | params=118.11M | checkpoint keys=171
Trainable parameters: 118.40M / 118.40M (100.0%)


In [5]:
# Cell 5 — BM25 hard-negative mining + Model 1 cross-encoder training
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

def bm25_clean(text):
    return re.sub(r"\s+", " ", str(text)).strip()

class BM25Index:
    def __init__(self, docs, k1=1.35, b=0.72, max_features=300000):
        self.ids = list(docs.keys())
        self.texts = [bm25_clean(record_text(docs[x])) for x in self.ids]
        self.vectorizer = CountVectorizer(
            lowercase=True, stop_words="english",
            token_pattern=r"(?u)\b[a-zA-Z0-9][a-zA-Z0-9./_-]{1,}\b",
            ngram_range=(1,2), max_features=max_features,
            min_df=1, binary=False, dtype=np.float32
        )
        counts = self.vectorizer.fit_transform(self.texts).tocsr()
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.avgdl = float(self.doc_len.mean()) if len(self.doc_len) else 1.0
        df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(np.float32)
        n = len(self.ids)
        self.idf = np.log(1.0 + (n-df+0.5)/(df+0.5)).astype(np.float32)
        row_ids = np.repeat(np.arange(counts.shape[0]), np.diff(counts.indptr))
        tf = counts.data.astype(np.float32, copy=False)
        norm = k1 * (1.0-b+b*self.doc_len[row_ids]/max(self.avgdl,1.0))
        weighted = self.idf[counts.indices] * (tf*(k1+1.0)/(tf+norm))
        self.matrix = sparse.csr_matrix(
            (weighted.astype(np.float32), counts.indices.copy(), counts.indptr.copy()),
            shape=counts.shape
        )
        self.id_to_row = {x:i for i,x in enumerate(self.ids)}
    def search(self, query_record, top_n=100, exclude_id=None):
        q = self.vectorizer.transform([bm25_clean(record_text(query_record))])
        scores = np.asarray((self.matrix @ q.T).toarray()).ravel()
        if exclude_id in self.id_to_row:
            scores[self.id_to_row[exclude_id]] = -1e9
        k = min(top_n, len(scores))
        if k == 0: return []
        idx = np.argpartition(-scores,k-1)[:k]
        idx = idx[np.argsort(-scores[idx])]
        return [(self.ids[i],float(scores[i])) for i in idx]

print("Building Train BM25 index:", stamp())
t_bm=time.time()
bm25_train = BM25Index(train_candidates)
print("Train BM25 index built | vocab:",len(bm25_train.vectorizer.vocabulary_),
      "| minutes:",round((time.time()-t_bm)/60,2))

train_examples=[]
missing_positive=0
for qid,qrec in train_queries.items():
    positives=[str(x) for x in qrec.get("relevant_candidates",[])
               if str(x) in train_candidates and str(x)!=qid]
    if not positives:
        missing_positive += 1
        continue
    ranked=bm25_train.search(qrec,top_n=HARD_NEGATIVE_POOL,exclude_id=qid)
    pset=set(positives)
    negs=[cid for cid,_ in ranked if cid not in pset]
    if len(negs)<TRAIN_NEGATIVES:
        negs += [cid for cid in train_candidates if cid not in pset and cid!=qid and cid not in negs]
    if negs:
        train_examples.append({"qid":qid,"positive_ids":positives,
                               "hard_negative_ids":negs[:max(3*TRAIN_NEGATIVES,30)]})
print(f"Train groups: {len(train_examples):,} | no usable positive: {missing_positive:,}")
assert len(train_examples)>0, "No usable positive query groups; check ID format and candidate pool."

_SIDE_CACHE={}
def encode_side(record, limit):
    key=(id(record),limit)
    hit=_SIDE_CACHE.get(key)
    if hit is None:
        hit=_encode_side_uncached(record,limit)
        _SIDE_CACHE[key]=hit
    return hit

def _encode_side_uncached(record, limit):
    ids=tokenizer.encode(record_text(record))
    if not ids: ids=[tokenizer.unk_token_id]
    if len(ids)>limit:
        # Head+tail keeps facts, statutes, and concluding holdings.
        head=int(limit*0.68)
        ids=ids[:head]+ids[-(limit-head):]
    return ids

def make_pair_ids(qrec, crec):
    q=encode_side(qrec,QUERY_TOKENS)
    c=encode_side(crec,CANDIDATE_TOKENS)
    ids=[tokenizer.bos_token_id]+q+[tokenizer.eos_token_id]+c+[tokenizer.eos_token_id]
    return ids[:PAIR_MAX_LEN]

def collate_pairs(pairs):
    rows=[make_pair_ids(q,c) for q,c in pairs]
    width=max(len(x) for x in rows)
    x=torch.full((len(rows),width),tokenizer.pad_token_id,dtype=torch.long)
    valid=torch.zeros((len(rows),width),dtype=torch.bool)
    for i,row in enumerate(rows):
        x[i,:len(row)]=torch.tensor(row,dtype=torch.long)
        valid[i,:len(row)]=True
    return x.to(DEVICE),valid.to(DEVICE)

def make_training_group(ex, pos_id=None):
    # Same objective as before: 1 positive vs TRAIN_NEGATIVES BM25 hard negatives.
    q=train_queries[ex["qid"]]
    if pos_id is None:
        pos_id=random.choice(ex["positive_ids"])
    neg_ids=random.sample(ex["hard_negative_ids"],k=min(TRAIN_NEGATIVES,len(ex["hard_negative_ids"])))
    return [(q,train_candidates[pos_id])] + [(q,train_candidates[cid]) for cid in neg_ids]

def build_epoch_groups():
    # Queries have several gold candidates; use up to POS_PER_QUERY distinct ones per epoch
    # instead of a single random positive (more supervision from the same data).
    groups=[]
    for ex in train_examples:
        pos=random.sample(ex["positive_ids"],k=min(POS_PER_QUERY,len(ex["positive_ids"])))
        groups.extend((ex,p) for p in pos)
    random.shuffle(groups)
    return groups

groups_per_epoch = sum(min(POS_PER_QUERY,len(ex["positive_ids"])) for ex in train_examples)

class WeightEMA:
    def __init__(self, module, decay):
        self.decay=decay; self.n=0
        self.params={n:p for n,p in module.named_parameters() if p.requires_grad}
        self.shadow={n:p.detach().clone().float() for n,p in self.params.items()}
    @torch.no_grad()
    def update(self):
        self.n+=1
        d=min(self.decay,(1+self.n)/(10+self.n))
        for n,p in self.params.items():
            self.shadow[n].mul_(d).add_(p.detach().float(),alpha=1-d)
    @contextmanager
    def scope(self):
        backup={n:p.detach().clone() for n,p in self.params.items()}
        with torch.no_grad():
            for n,p in self.params.items(): p.copy_(self.shadow[n].to(p.dtype))
        try:
            yield
        finally:
            with torch.no_grad():
                for n,p in self.params.items(): p.copy_(backup[n])
            del backup

# Differential learning rates for stable full fine-tuning.
# Head > top blocks > lower blocks > token embeddings.
param_groups = [{"params": list(pair_model.score.parameters()), "lr": LEARNING_RATE*HEAD_LR_MULT}]
param_groups.append({"params": list(model.token_emb.parameters()), "lr": EMBEDDING_LR})
param_groups.append({"params": list(model.final_ln.parameters()), "lr": LEARNING_RATE})
for depth, block in enumerate(reversed(list(model.blocks))):
    param_groups.append({
        "params": list(block.parameters()),
        "lr": LEARNING_RATE * (LAYER_LR_DECAY ** depth)
    })

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(groups_per_epoch / ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.10 + 0.90 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
ema = WeightEMA(pair_model, EMA_DECAY)

print("\nTraining plan")
print(f"  queries={len(train_examples):,} | groups/epoch={groups_per_epoch:,} | optimizer steps={total_steps:,} | grad accumulation={ACCUM_STEPS}")
print(f"  pairs/forward={1+TRAIN_NEGATIVES} | epochs={EPOCHS} | trainable={trainable/1e6:.1f}M")
print("  Runtime is workload/GPU dependent; first epoch will provide a measured estimate.")
print("  Optimizer group LRs:",[round(g["lr"],8) for g in param_groups])


Building Train BM25 index: 2026-11-07 08:02:41 IST
Train BM25 index built | vocab: 300000 | minutes: 0.87
Train groups: 817 | no usable positive: 10

Training plan
  queries=817 | groups/epoch=1,513 | optimizer steps=3,785 | grad accumulation=2
  pairs/forward=6 | epochs=5 | trainable=118.4M
  Runtime is workload/GPU dependent; first epoch will provide a measured estimate.
  Optimizer group LRs: [7e-07, 1e-08, 9e-08, 9e-08, 7e-08, 6e-08, 5e-08, 5e-08, 4e-08, 3e-08, 3e-08, 2e-08, 2e-08, 2e-08, 1e-08, 1e-08, 1e-08]


In [6]:
# Cell 6 — Dev evaluation, epoch-wise checkpoint selection, and training loop
@torch.no_grad()
def score_pairs(pair_list,batch_size=EVAL_BATCH_SIZE):
    pair_model.eval()
    out=[]
    for start in range(0,len(pair_list),batch_size):
        x,valid=collate_pairs(pair_list[start:start+batch_size])
        with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=USE_AMP):
            logits=pair_model(x,valid)
        out.extend(logits.float().cpu().numpy().tolist())
    return np.asarray(out,dtype=np.float32)

def official_metrics(queries,predictions):
    tp=fp=fn=0
    for qid,q in queries.items():
        gold=set(str(x) for x in q.get("relevant_candidates",[]))
        pred=set(predictions.get(qid,[]))
        tp+=len(gold & pred); fp+=len(pred-gold); fn+=len(gold-pred)
    p=tp/(tp+fp) if tp+fp else 0.0
    r=tp/(tp+fn) if tp+fn else 0.0
    f1=2*p*r/(p+r) if p+r else 0.0
    return {"micro_f1_at_7":f1,"micro_precision_at_7":p,"micro_recall_at_7":r,
            "TP":tp,"FP":fp,"FN":fn}

print("Building Dev BM25 index:",stamp())
t_dev_bm=time.time()
bm25_dev=BM25Index(dev_candidates)
print("Dev index minutes:",round((time.time()-t_dev_bm)/60,2))

def fuse_predictions(cache,alpha):
    out={}
    for qid,(ids,lex,neu) in cache.items():
        order=np.argsort(-(neu+alpha*lex))
        out[qid]=[ids[j] for j in order[:TOP_K]]
    return out

def tune_alpha(queries,cache):
    best_a,best_m=None,None
    for a in FUSION_GRID:
        m=official_metrics(queries,fuse_predictions(cache,a))
        if best_m is None or m["micro_f1_at_7"]>best_m["micro_f1_at_7"]:
            best_a,best_m=a,m
    return best_a,best_m

def retrieve_and_rerank_with_index(queries,candidates,index,split_name):
    reranked={}; bm25_only={}; cache={}
    for qi,(qid,qrec) in enumerate(queries.items(),1):
        ranked=index.search(qrec,top_n=RERANK_TOP_N,exclude_id=qid)
        if not ranked:
            raise RuntimeError(f"No candidates retrieved for {split_name} query {qid}")
        bm25_only[qid]=[cid for cid,_ in ranked[:TOP_K]]
        pair_list=[(qrec,candidates[cid]) for cid,_ in ranked]
        neural=score_pairs(pair_list)
        lexical=np.asarray([s for _,s in ranked],dtype=np.float32)
        lexical=(lexical-lexical.mean())/(lexical.std()+1e-6)
        neural=(neural-neural.mean())/(neural.std()+1e-6)
        cache[qid]=([cid for cid,_ in ranked],lexical,neural)
        fused=neural+0.12*lexical
        order=np.argsort(-fused)
        reranked[qid]=[ranked[j][0] for j in order[:TOP_K]]
        if qi%10==0 or qi==len(queries):
            print(f"[{stamp()}] {split_name} reranked {qi}/{len(queries)}",flush=True)
    return reranked,bm25_only,cache

# BM25 baseline Dev score (computed once).
dev_bm25_predictions={}
for qi,(qid,qrec) in enumerate(dev_queries.items(),1):
    ranked=bm25_dev.search(qrec,top_n=TOP_K,exclude_id=qid)
    dev_bm25_predictions[qid]=[cid for cid,_ in ranked]
bm25_dev_metrics=official_metrics(dev_queries,dev_bm25_predictions)
print("BM25-only Dev:",json.dumps(bm25_dev_metrics,indent=2))

best_dev_f1 = -1.0
best_epoch = None
BEST_ALPHA = 0.12
history = []
epoch_times = []
training_start = time.time()
best_path = CKPT_DIR / "best_dev_f1.pt"

for epoch in range(EPOCHS):
    pair_model.train()
    epoch_groups = build_epoch_groups()
    running = 0.0
    steps = 0
    epoch_start = time.time()
    optimizer.zero_grad(set_to_none=True)

    # Each query group is forwarded/backpropagated separately; gradients accumulate
    # across ACCUM_STEPS groups. This keeps activation memory manageable on a T4.
    n_groups = len(epoch_groups)
    for start in range(0, n_groups, ACCUM_STEPS):
        group_chunk = epoch_groups[start:start+ACCUM_STEPS]
        optimizer.zero_grad(set_to_none=True)
        chunk_loss_value = 0.0

        for ex, pos_id in group_chunk:
            gp = make_training_group(ex, pos_id)
            x, valid = collate_pairs(gp)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = pair_model(x, valid)
            loss = F.cross_entropy(logits.float().unsqueeze(0),
                                   torch.zeros(1, dtype=torch.long, device=DEVICE))
            scaler.scale(loss / len(group_chunk)).backward()
            chunk_loss_value += float(loss.detach())

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in pair_model.parameters() if p.requires_grad], GRAD_CLIP
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update()

        running += chunk_loss_value / len(group_chunk)
        steps += 1
        if steps % 25 == 0 or start + len(group_chunk) >= n_groups:
            elapsed = time.time() - epoch_start
            rate = steps / max(elapsed, 1e-6)
            remaining = (steps_per_epoch - steps) / max(rate, 1e-6)
            print(f"[{stamp()}] epoch {epoch+1}/{EPOCHS} step {steps}/{steps_per_epoch} "
                  f"loss={running/steps:.4f} elapsed={elapsed/60:.1f}m "
                  f"ETA={remaining/60:.1f}m", flush=True)

    epoch_loss = running / max(steps, 1)
    epoch_train_seconds = time.time() - epoch_start
    epoch_times.append(epoch_train_seconds)

    # Evaluate and checkpoint the EMA-averaged weights (raw weights are restored after the scope).
    with ema.scope():
        # Save epoch checkpoint directly in the flat experiment folder.
        epoch_path = CKPT_DIR / f"epoch_{epoch+1}.pt"
        torch.save({
            "model": model.state_dict(),
            "scorer": pair_model.score.state_dict(),
            "epoch": epoch+1,
            "train_loss": epoch_loss,
            "base_checkpoint": CKPT_PATH,
            "tokenizer": TOKENIZER_PATH,
            "fine_tuning": "FULL",
            "trainable_parameters": trainable,
        }, epoch_path)

        dev_reranked, _, dev_cache = retrieve_and_rerank_with_index(
            dev_queries, dev_candidates, bm25_dev, f"Dev epoch {epoch+1}"
        )
        epoch_alpha, dev_metrics = tune_alpha(dev_queries, dev_cache)
        dev_metrics["fusion_alpha"] = epoch_alpha
        row = {
            "epoch": epoch+1,
            "train_loss": epoch_loss,
            "train_minutes": epoch_train_seconds/60,
            "dev_metrics": dev_metrics
        }
        history.append(row)
        print(f"\nEPOCH {epoch+1} | loss={epoch_loss:.5f} | train_min={epoch_train_seconds/60:.1f}")
        print("Dev reranker:", json.dumps(dev_metrics, indent=2))

        if dev_metrics["micro_f1_at_7"] > best_dev_f1:
            best_dev_f1 = dev_metrics["micro_f1_at_7"]
            best_epoch = epoch+1
            BEST_ALPHA = epoch_alpha
            torch.save({
                "model": model.state_dict(),
                "scorer": pair_model.score.state_dict(),
                "epoch": epoch+1,
                "train_loss": epoch_loss,
                "dev_metrics": dev_metrics,
                "base_checkpoint": CKPT_PATH,
                "tokenizer": TOKENIZER_PATH,
                "fine_tuning": "FULL",
                "trainable_parameters": trainable,
            }, best_path)
            print("NEW BEST DEV CHECKPOINT:", best_path, "| Dev F1:", best_dev_f1)

    with open(OUTPUT_DIR/"training_history.json", "w") as f:
        json.dump({"history": history, "best_epoch": best_epoch,
                   "best_dev_f1": best_dev_f1}, f, indent=2)

    elapsed_total = time.time() - training_start
    completed = epoch+1
    avg_epoch = elapsed_total / completed
    eta = (EPOCHS-completed) * avg_epoch
    print(f"Elapsed total={elapsed_total/60:.1f} min | remaining training+dev ETA≈{eta/60:.1f} min")
    if epoch+1-best_epoch >= PATIENCE:
        print(f"Early stopping: no Dev improvement for {PATIENCE} epochs.")
        break


print("\nTraining complete:",stamp(),"| total min:",round((time.time()-training_start)/60,2))
print("Best epoch:",best_epoch,"| best Dev F1:",best_dev_f1)
assert best_path.exists(), "No best checkpoint was saved."

# Restore the best checkpoint before final Dev comparison and Test inference.
best_obj=torch.load(best_path,map_location="cpu",weights_only=False)
model.load_state_dict(best_obj["model"],strict=True)
pair_model.score.load_state_dict(best_obj["scorer"],strict=True)
pair_model.eval()
print("Restored best FULL-FT (EMA) checkpoint | epoch:",best_obj["epoch"],
      "| Dev F1:",best_obj["dev_metrics"]["micro_f1_at_7"])


Building Dev BM25 index: 2026-11-07 08:04:31 IST
Dev index minutes: 0.29
BM25-only Dev: {
  "micro_f1_at_7": 0.330183435241801,
  "micro_precision_at_7": 0.3595641646489104,
  "micro_recall_at_7": 0.3052415210688592,
  "TP": 297,
  "FP": 529,
  "FN": 676
}


/tmp/ipykernel_49/4082292570.py:113: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[2026-11-07 08:05:12 IST] epoch 1/5 step 25/757 loss=1.7854 elapsed=0.3m ETA=10.1m
[2026-11-07 08:05:29 IST] epoch 1/5 step 50/757 loss=1.8019 elapsed=0.6m ETA=8.8m
[2026-11-07 08:05:42 IST] epoch 1/5 step 75/757 loss=1.8058 elapsed=0.8m ETA=7.7m
[2026-11-07 08:05:55 IST] epoch 1/5 step 100/757 loss=1.8034 elapsed=1.1m ETA=7.0m
[2026-11-07 08:06:08 IST] epoch 1/5 step 125/757 loss=1.7993 elapsed=1.3m ETA=6.4m
[2026-11-07 08:06:20 IST] epoch 1/5 step 150/757 loss=1.8009 elapsed=1.5m ETA=6.0m
[2026-11-07 08:06:34 IST] epoch 1/5 step 175/757 loss=1.8013 elapsed=1.7m ETA=5.7m
[2026-11-07 08:06:47 IST] epoch 1/5 step 200/757 loss=1.7979 elapsed=1.9m ETA=5.3m
[2026-11-07 08:06:59 IST] epoch 1/5 step 225/757 loss=1.7990 elapsed=2.1m ETA=5.0m
[2026-11-07 08:07:12 IST] epoch 1/5 step 250/757 loss=1.8040 elapsed=2.3m ETA=4.7m
[2026-11-07 08:07:24 IST] epoch 1/5 step 275/757 loss=1.8013 elapsed=2.5m ETA=4.5m
[2026-11-07 08:07:35 IST] epoch 1/5 step 300/757 loss=1.8041 elapsed=2.7m ETA=4.2m
[2026-

In [7]:
# Cell 7 — Final Dev comparison + Test inference + validated submission ZIP
# Re-evaluate the restored best checkpoint on Dev.
dev_reranked,dev_bm25_check,dev_cache=retrieve_and_rerank_with_index(
    dev_queries,dev_candidates,bm25_dev,"Final Dev (best checkpoint)"
)
FINAL_ALPHA,dev_rerank_metrics=tune_alpha(dev_queries,dev_cache)
dev_reranked=fuse_predictions(dev_cache,FINAL_ALPHA)
dev_rerank_metrics["fusion_alpha"]=FINAL_ALPHA
print("Tuned lexical-fusion alpha on Dev:",FINAL_ALPHA)
dev_bm25_metrics=official_metrics(dev_queries,dev_bm25_check)
print("\nFINAL DEV")
print("BM25:",json.dumps(dev_bm25_metrics,indent=2))
print("Model 1 full-finetuned reranker:",json.dumps(dev_rerank_metrics,indent=2))

(OUTPUT_DIR/"dev_predictions_model1.json").write_text(
    json.dumps(dev_reranked,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_DIR/"dev_metrics_model1.json").write_text(
    json.dumps({"bm25":dev_bm25_metrics,"model1_full_ft_cross_encoder":dev_rerank_metrics,
                "best_epoch":best_epoch,"training_history":history},indent=2),encoding="utf-8")

# Only use each split's own candidate pool. Select on Dev, not Test labels.
print("\nBuilding Test BM25 index:",stamp())
t_test_bm=time.time()
bm25_test=BM25Index(test_candidates)
print("Test index minutes:",round((time.time()-t_test_bm)/60,2))

_,test_bm25,test_cache=retrieve_and_rerank_with_index(
    test_queries,test_candidates,bm25_test,"Test"
)
test_reranked=fuse_predictions(test_cache,FINAL_ALPHA)
if dev_rerank_metrics["micro_f1_at_7"] >= dev_bm25_metrics["micro_f1_at_7"]:
    test_predictions=test_reranked
    selected_method="BM25 + Model 1 cross-encoder reranking"
else:
    test_predictions=test_bm25
    selected_method="BM25-only (Model 1 reranking did not improve Dev F1)"
print("Selected Test method:",selected_method)

assert set(test_predictions)==set(test_queries)
for qid,ids in test_predictions.items():
    assert len(ids)==TOP_K and len(set(ids))==TOP_K, (qid,ids)
    assert all(cid in test_candidates for cid in ids), qid

prediction_path=OUTPUT_DIR/"prediction.json"
prediction_path.write_text(json.dumps(test_predictions,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
with open(prediction_path,encoding="utf-8") as f:
    checked=json.load(f)
assert checked==test_predictions

zip_path=OUTPUT_DIR/"phase7_model1_full_ft_submission.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(prediction_path,arcname="prediction.json")
with zipfile.ZipFile(zip_path,"r") as z:
    assert z.namelist()==["prediction.json"]
    assert json.loads(z.read("prediction.json").decode("utf-8"))==test_predictions

summary={
    "experiment":"Phase 7 Model 1 Full Fine-Tuning",
    "selected_method":selected_method,
    "best_epoch":best_epoch,
    "best_dev_f1":best_dev_f1,
    "final_dev_bm25":dev_bm25_metrics,
    "final_dev_model1":dev_rerank_metrics,
    "test_queries":len(test_predictions),
    "submission_zip":str(zip_path),
    "finished_at":stamp()
}
json.dump(summary,open(OUTPUT_DIR/"experiment_summary.json","w"),indent=2)
print("\nSUBMISSION READY")
print(json.dumps(summary,indent=2))
print("ZIP:",zip_path,"| MB:",round(zip_path.stat().st_size/1024**2,3))


[2026-11-07 08:28:07 IST] Final Dev (best checkpoint) reranked 10/118
[2026-11-07 08:28:15 IST] Final Dev (best checkpoint) reranked 20/118
[2026-11-07 08:28:23 IST] Final Dev (best checkpoint) reranked 30/118
[2026-11-07 08:28:31 IST] Final Dev (best checkpoint) reranked 40/118
[2026-11-07 08:28:40 IST] Final Dev (best checkpoint) reranked 50/118
[2026-11-07 08:28:48 IST] Final Dev (best checkpoint) reranked 60/118
[2026-11-07 08:28:56 IST] Final Dev (best checkpoint) reranked 70/118
[2026-11-07 08:29:04 IST] Final Dev (best checkpoint) reranked 80/118
[2026-11-07 08:29:12 IST] Final Dev (best checkpoint) reranked 90/118
[2026-11-07 08:29:20 IST] Final Dev (best checkpoint) reranked 100/118
[2026-11-07 08:29:28 IST] Final Dev (best checkpoint) reranked 110/118
[2026-11-07 08:29:35 IST] Final Dev (best checkpoint) reranked 118/118
Tuned lexical-fusion alpha on Dev: 10.0

FINAL DEV
BM25: {
  "micro_f1_at_7": 0.330183435241801,
  "micro_precision_at_7": 0.3595641646489104,
  "micro_recal

In [8]:
import subprocess
import sys

In [9]:
# Cell 8 — Professor's exact evaluate_phase7.py + pre-Codabench validation
# This writes the supplied evaluator verbatim to the experiment output directory.
EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).\n\nIt uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /\nMicro-Recall@7 definitions as the Codabench scorer.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nK = 7\n\n\ndef reject_constant(value: str) -> None:\n    raise ValueError(f"Non-standard JSON numeric value \'{value}\' is not allowed.")\n\n\ndef pairs_no_duplicates(pairs):\n    result = {}\n    duplicates = []\n    for key, value in pairs:\n        if key in result:\n            duplicates.append(key)\n        result[key] = value\n    if duplicates:\n        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")\n    return result\n\n\ndef load_json(path: Path) -> Any:\n    try:\n        with path.open("r", encoding="utf-8") as f:\n            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n    except json.JSONDecodeError as exc:\n        raise ValueError(\n            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"\n        ) from exc\n\n\ndef load_truth(path: Path) -> dict[str, list[str]]:\n    data = load_json(path)\n    if not isinstance(data, dict) or not data:\n        raise ValueError("Truth must be a non-empty JSON object.")\n    out: dict[str, list[str]] = {}\n    for query_id, value in data.items():\n        if isinstance(value, dict) and "relevant_candidates" in value:\n            labels = value["relevant_candidates"]\n        else:\n            labels = value\n        if not isinstance(query_id, str) or not query_id.strip():\n            raise ValueError("Truth contains an empty/invalid query ID.")\n        if not isinstance(labels, list):\n            raise ValueError(f"Truth for query \'{query_id}\' must contain a list of relevant candidate IDs.")\n        cleaned=[]; seen=set()\n        for idx,candidate_id in enumerate(labels):\n            if not isinstance(candidate_id,str) or not candidate_id.strip():\n                continue\n            if candidate_id in seen:\n                raise ValueError(f"Truth for query \'{query_id}\' contains duplicate candidate ID \'{candidate_id}\'.")\n            seen.add(candidate_id); cleaned.append(candidate_id)\n        out[query_id]=cleaned\n    return out\n\n\ndef load_candidate_ids(path: Path) -> set[str]:\n    data = load_json(path)\n    if not isinstance(data, list) or not data:\n        raise ValueError("candidate_ids must be a non-empty JSON list.")\n    ids=[]; seen=set()\n    for idx,v in enumerate(data):\n        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")\n        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID \'{v}\'.")\n        seen.add(v); ids.append(v)\n    return set(ids)\n\n\ndef load_prediction(path: Path) -> Any:\n    return load_json(path)\n\n\ndef validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:\n    errors=[]\n    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]\n    gold_ids=set(gold); pred_ids=set(pred)\n    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID \'{q}\'.")\n    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID \'{q}\'.")\n    for q in sorted(gold_ids & pred_ids):\n        vals=pred[q]\n        if not isinstance(vals,list):\n            errors.append(f"Query \'{q}\': prediction must be a list."); continue\n        if len(vals)!=K: errors.append(f"Query \'{q}\': expected exactly {K} candidates, got {len(vals)}.")\n        seen=set()\n        for i,c in enumerate(vals):\n            if not isinstance(c,str) or not c.strip(): errors.append(f"Query \'{q}\': index {i} is not a non-empty string."); continue\n            if c in seen: errors.append(f"Query \'{q}\': duplicate candidate ID \'{c}\'.")\n            seen.add(c)\n            if c not in candidates: errors.append(f"Query \'{q}\': candidate ID \'{c}\' is outside the candidate pool.")\n    return errors\n\n\ndef score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:\n    tp=fp=fn=0\n    for q,labels in gold.items():\n        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)\n    precision=tp/(tp+fp) if tp+fp else 0.0\n    recall=tp/(tp+fn) if tp+fn else 0.0\n    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0\n    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}\n\n\ndef parse_args():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")\n    p.add_argument("--prediction",type=Path,required=True)\n    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")\n    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")\n    return p.parse_args()\n\n\ndef main():\n    a=parse_args()\n    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)\n    errs=validate(pred,gold,candidates)\n    if errs:\n        print("LOCAL VALIDATION FAILED")\n        for i,e in enumerate(errs,1): print(f"  {i}. {e}")\n        print(f"Total validation errors: {len(errs)}")\n        raise SystemExit(1)\n    result=score(gold,pred)\n    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}\n    print("Validation passed.")\n    print(f"Micro-F1@7: {scores[\'primary_metric_1\']:.10f}")\n    print(f"Micro-Precision@7: {scores[\'secondary_metric_1\']:.10f}")\n    print(f"Micro-Recall@7: {scores[\'secondary_metric_2\']:.10f}")\n    print(f"TP={result[\'tp\']} FP={result[\'fp\']} FN={result[\'fn\']}")\n    if a.output:\n        a.output.parent.mkdir(parents=True,exist_ok=True)\n        with a.output.open(\'w\',encoding=\'utf-8\') as f: json.dump(scores,f,indent=2); f.write(\'\\n\')\n        print(f"Wrote {a.output}")\n\nif __name__==\'__main__\': main()\n'
EVALUATOR_PATH = OUTPUT_DIR / "evaluate_phase7.py"
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding="utf-8")
print("Official evaluator saved:", EVALUATOR_PATH)

# Candidate ID files in the exact format expected by the professor's evaluator.
DEV_CANDIDATE_IDS_PATH = OUTPUT_DIR / "dev_candidate_ids.json"
DEV_CANDIDATE_IDS_PATH.write_text(
    json.dumps(list(dev_candidates.keys()), ensure_ascii=False, indent=2), encoding="utf-8"
)
DEV_PREDICTION_PATH = OUTPUT_DIR / "dev_predictions_model1.json"
DEV_TRUTH_PATH = OUTPUT_DIR / "dev_truth.json"
DEV_TRUTH_PATH.write_text(json.dumps(dev_queries, ensure_ascii=False), encoding="utf-8")

# Run the professor's CLI evaluator on labelled Dev data.
DEV_EVAL_OUTPUT = OUTPUT_DIR / "official_dev_scores.json"
cmd = [
    sys.executable, str(EVALUATOR_PATH),
    "--truth", str(DEV_TRUTH_PATH),
    "--prediction", str(DEV_PREDICTION_PATH),
    "--candidate-ids", str(DEV_CANDIDATE_IDS_PATH),
    "--output", str(DEV_EVAL_OUTPUT),
]
print("\\nRunning professor's evaluator on Dev:", stamp())
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Professor's official evaluator FAILED on Dev predictions.")
print("Official Dev evaluation passed. Scores saved:", DEV_EVAL_OUTPUT)

# Pre-Codabench Test validation: test labels are private, so do NOT report Test F1.
# Use the evaluator's exact validate() function with an empty-label truth mapping
# solely to verify query coverage, exactly 7 IDs, uniqueness, string IDs, and pool membership.
TEST_CANDIDATE_IDS_PATH = OUTPUT_DIR / "test_candidate_ids.json"
TEST_CANDIDATE_IDS_PATH.write_text(
    json.dumps(list(test_candidates.keys()), ensure_ascii=False, indent=2), encoding="utf-8"
)
spec = __import__("importlib.util").util.spec_from_file_location("evaluate_phase7", EVALUATOR_PATH)
evaluator = __import__("importlib.util").util.module_from_spec(spec)
spec.loader.exec_module(evaluator)
test_truth_for_validation = {str(qid): [] for qid in test_queries}
test_candidate_set = evaluator.load_candidate_ids(TEST_CANDIDATE_IDS_PATH)
test_prediction_obj = evaluator.load_prediction(prediction_path)
test_errors = evaluator.validate(test_prediction_obj, test_truth_for_validation, test_candidate_set)
if test_errors:
    print("TEST SUBMISSION VALIDATION FAILED")
    for i, err in enumerate(test_errors, 1):
        print(f"  {i}. {err}")
    raise RuntimeError("Test submission failed professor's structural validation.")
print("TEST SUBMISSION STRUCTURAL VALIDATION PASSED")
print("Checked: exact Test query IDs | exactly 7 IDs/query | no duplicates | all IDs in Test candidate pool.")
print("No Test score is calculated because Test relevance labels are private.")
print("Final ZIP to upload:", zip_path)


Official evaluator saved: /kaggle/working/phase7_model1_full_ft/evaluate_phase7.py
\nRunning professor's evaluator on Dev: 2026-11-07 08:33:34 IST
Validation passed.
Micro-F1@7: 0.3324068927
Micro-Precision@7: 0.3619854722
Micro-Recall@7: 0.3072970195
TP=299 FP=527 FN=674
Wrote /kaggle/working/phase7_model1_full_ft/official_dev_scores.json

Official Dev evaluation passed. Scores saved: /kaggle/working/phase7_model1_full_ft/official_dev_scores.json
TEST SUBMISSION STRUCTURAL VALIDATION PASSED
Checked: exact Test query IDs | exactly 7 IDs/query | no duplicates | all IDs in Test candidate pool.
No Test score is calculated because Test relevance labels are private.
Final ZIP to upload: /kaggle/working/phase7_model1_full_ft/phase7_model1_full_ft_submission.zip


## Run checklist and interpretation

1. Select **GPU T4** in Kaggle.
2. Attach the Phase-1 Model 1 checkpoint/tokenizer dataset. Confirm the printed paths point to the matching `enc_full_best*.pt` and `tokenizer (1).json`.
3. Ensure the Kaggle Secret is named `HF_TOKEN` if the Phase-7 dataset is private.
4. Run all cells from top to bottom.
5. This experiment fully fine-tunes the Model 1 token embeddings, all 14 transformer blocks, final normalization, and ranking head. Gradient checkpointing and two-query gradient accumulation are enabled for T4 memory safety.
6. All output files are written directly under `/kaggle/working/phase7_model1_full_ft/` with **no subfolders**.
7. The notebook selects checkpoints and fusion weight using labelled Dev Micro-F1@7 only. Test labels are never used.
8. Upload the generated `phase7_model1_full_ft_submission.zip` to Codabench.
9. Full fine-tuning is an experiment, not a guaranteed score improvement. Compare its official Test score against the previous best of 0.3473.
